# 01 Data Cleaning and Mapping

This notebook constructs the final occupation-year panel by merging German KldB occupation-level labour market data with AI exposure measures.

**Output:** `data/processed/final_dataset_ai_exposure_controls.csv`

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "outputs" / "figures"
TABLES = PROJECT_ROOT / "outputs" / "tables"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd

## 1. Load occupation-year panel

Place the cleaned Engpassanalyse/control file in:

`data/raw/panel_dataset_controls.csv`

In [ ]:
panel_path = DATA_RAW / "panel_dataset_controls.csv"
panel = pd.read_csv(panel_path)

panel.columns = (
    panel.columns
    .astype(str)
    .str.strip()
)

print(panel.shape)
panel.head()

## 2. Clean variable types

In [ ]:
numeric_cols = [
    "kldb3", "year", "bestand_absolut", "vakanz_tage",
    "relation", "log_bestand", "log_vakanz", "log_relation"
]

for col in numeric_cols:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

panel["kldb3"] = panel["kldb3"].astype("Int64")
panel["year"] = panel["year"].astype("Int64")

panel = panel.dropna(subset=numeric_cols).copy()

print(panel.shape)
panel.head()

## 3. Load AI exposure data

Place the AIOE data appendix in:

`data/raw/AIOE_DataAppendix.xlsx`

In [ ]:
ai_path = DATA_RAW / "AIOE_DataAppendix.xlsx"

ai = pd.read_excel(ai_path, sheet_name="Appendix A")

ai = ai.rename(columns={
    "SOC Code": "soc",
    "Occupation Title": "occupation_title",
    "AIOE": "ai_exposure"
})

ai = ai[["soc", "occupation_title", "ai_exposure"]].copy()

ai["soc"] = ai["soc"].astype(str).str.replace("-", "", regex=False)
ai["soc_major"] = ai["soc"].str[:2]
ai["ai_exposure"] = pd.to_numeric(ai["ai_exposure"], errors="coerce")

ai = ai.dropna(subset=["soc_major", "ai_exposure"]).copy()

ai_major = (
    ai.groupby("soc_major", as_index=False)["ai_exposure"]
    .mean()
)

ai_major.head()

## 4. KldB 3-digit to SOC major crosswalk

In [ ]:
crosswalk = {
    111:45,112:45,113:45,115:45,117:45,121:45,122:39,
    211:47,212:51,221:51,222:51,223:51,231:51,232:27,234:51,
    241:51,242:51,243:51,244:51,245:51,
    251:17,252:17,261:17,262:17,263:17,271:19,272:17,273:13,
    281:51,282:51,283:51,291:51,292:51,293:35,
    311:17,312:17,321:47,322:47,331:47,332:47,333:47,
    341:49,342:47,343:51,
    411:15,412:19,413:19,414:19,421:19,422:19,423:19,
    431:15,432:15,433:15,434:15,
    511:53,512:53,513:53,514:53,515:53,516:43,521:53,522:53,524:53,525:53,
    531:33,532:33,533:33,541:37,
    611:41,612:41,613:41,621:41,622:41,623:41,624:41,
    631:39,632:35,633:35,634:39,
    713:13,714:43,715:13,721:13,722:13,723:13,731:23,732:43,733:43,
    811:31,812:29,813:29,814:29,815:29,816:29,817:29,818:29,
    821:31,822:29,823:39,825:29,
    831:21,832:39,841:25,842:25,843:25,844:25,845:25,
    912:19,913:19,921:13,922:27,924:27,932:27,935:27,941:27,942:27
}

panel["soc_major"] = panel["kldb3"].map(crosswalk)
panel = panel.dropna(subset=["soc_major"]).copy()
panel["soc_major"] = panel["soc_major"].astype(int).astype(str)

print(panel[["kldb3", "soc_major"]].drop_duplicates().head())

## 5. Merge panel with AI exposure and collapse duplicates

In [ ]:
final = panel.merge(ai_major, on="soc_major", how="left")

print("Missing AI exposure share:", final["ai_exposure"].isna().mean())

final = (
    final.groupby(["kldb3", "year", "soc_major"], as_index=False)
    .agg({
        "bestand_absolut": "mean",
        "vakanz_tage": "mean",
        "relation": "mean",
        "log_bestand": "mean",
        "log_vakanz": "mean",
        "log_relation": "mean",
        "ai_exposure": "mean"
    })
)

final["post"] = (final["year"] >= 2023).astype(int)
final["ai_post"] = final["ai_exposure"] * final["post"]

print(final.shape)
final.head()

## 6. Save final analysis dataset

In [ ]:
out_path = DATA_PROCESSED / "final_dataset_ai_exposure_controls.csv"
final.to_csv(out_path, index=False)

print("Saved:", out_path)